![Banner](https://i.imgur.com/a3uAqnb.png)

# REINFORCE Algorithm Implementation - Homework Assignment

In this homework, you will implement the **REINFORCE** algorithm using PyTorch to train an agent to play the Lunar Lander game. This will involve understanding policy gradient methods, implementing Monte Carlo sampling, and optimizing the training process.

## 📌 Project Overview
- **Task**: Train an RL agent to land a spacecraft safely using policy gradients
- **Algorithm**: REINFORCE (Monte Carlo Policy Gradient)
- **Environment**: LunarLander-v3 from OpenAI Gym
- **Goal**: Achieve consistent successful landings (score > 200)

## 📚 Learning Objectives
By completing this assignment, you will:
- Understand policy gradient methods and the REINFORCE algorithm
- Implement Monte Carlo return calculation
- Apply policy gradient theorem for policy optimization
- Use return normalization for stable training
- Evaluate policy-based reinforcement learning agents
- Visualize training progress and agent behavior

## 🎯 Key Differences from Value-Based Methods
- **Direct Policy Optimization**: Unlike DQN which learns Q-values, REINFORCE directly optimizes the policy
- **Monte Carlo Returns**: Uses complete episode returns rather than bootstrapping
- **Stochastic Policy**: Outputs probability distributions over actions
- **Policy Gradient**: Updates policy parameters using gradient ascent on expected returns

## Lunar Lander

This environment is a classic rocket trajectory optimization problem. The landing pad is always at coordinates (0,0). The state is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

There are four discrete actions available:<br>
- 0: do nothing<br>
- 1: fire left orientation engine<br>
- 2: fire main engine<br>
- 3: fire right orientation engine<br>

After every step a reward is granted. The total reward of an episode is the sum of the rewards for all the steps within that episode.

For each step, the reward:

- is increased/decreased the closer/further the lander is to the landing pad.

- is increased/decreased the slower/faster the lander is moving.

- is decreased the more the lander is tilted (angle not horizontal).

- is increased by 10 points for each leg that is in contact with the ground.

- is decreased by 0.03 points each frame a side engine is firing.

- is decreased by 0.3 points each frame the main engine is firing.

The episode receive an additional reward of -100 or +100 points for crashing or landing safely respectively.

An episode is considered a solution if it scores at least 200 points.


You can read more the LunarLander environment [here](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

![LunarLander](https://gymnasium.farama.org/_images/lunar_lander.gif)

## Reinforce

REINFORCE is an elegant algorithm for maximizing the expected return. We sample a trajectory $\tau$ . If we get a high reward, we try to make it more likely. If we get a low reward, we try to make it less likely.

We just need a policy that maximizes the expected return and we can do this by Gradient Ascent on Policy parameters.

![Algorthm](https://i.imgur.com/G9Ybar2.png)

[Image Source](https://colab.research.google.com/github/huggingface/deep-rl-class/blob/master/notebooks/unit4/unit4.ipynb)

In [ ]:
# TODO: Install required packages (uncomment if needed):
# !pip install -q swig
# !pip install -q gym[box2d]
# !pip install -q pygame
# !pip install -q moviepy

# TODO: Import all necessary libraries:
#       - gymnasium for the environment (updated gym interface)
#       - torch, torch.nn, torch.optim for neural networks
#       - torch.nn.functional for activation functions
#       - torch.distributions for probability distributions
#       - math, random, numpy for utilities
#       - matplotlib for plotting
#       - collections.deque for efficient data structures

import gymnasium as gym  # Updated gym interface
import math
import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
from torch.distributions import Categorical

# TODO: Set random seeds for reproducibility
# TODO: Check if GPU is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1️⃣ Environment Setup

**Task**: Create and explore the Lunar Lander environment.

**Requirements**:
- Initialize the LunarLander-v3 environment
- Understand the state space (8-dimensional vector)
- Understand the action space (4 discrete actions)
- Explore the reward structure and episode termination conditions

In [ ]:
# TODO: Create the Lunar Lander environment
#       - Use gym.make("LunarLander-v3")
#       - Print environment information

env = gym.make("LunarLander-v3")

# TODO: Print environment details:
#       - Action space size
#       - Observation space shape
#       - Reset environment and print initial state

print(f"Action space size: {env.action_space.n}")
print(f"Observation space shape: {env.observation_space.shape}")

# TODO: Reset environment and examine initial state
state, info = env.reset()
print(f"Initial state: {state}")
print(f"State dimension: {len(state)}")

# TODO: Explore one random step
action = env.action_space.sample()
next_state, reward, terminated, truncated, info = env.step(action)
print(f"Random action: {action}")
print(f"Reward: {reward}")
print(f"Terminated: {terminated}")

## 2️⃣ Policy Network Architecture

**Task**: Design the neural network that will represent our stochastic policy.

**Requirements**:
- Create a feedforward neural network that outputs action probabilities
- Input: 8-dimensional state vector (LunarLander observations)
- Output: 4-dimensional probability distribution over actions
- Use softmax activation for valid probability distribution
- Implement action sampling and log-probability calculation

In [ ]:
# TODO: Implement Policy class inheriting from nn.Module:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        """
        Initialize the policy network.
        
        Args:
            s_size: State space dimension (8 for LunarLander)
            a_size: Action space dimension (4 for LunarLander)
            h_size: Hidden layer size (64 is typically good)
        """
        super(Policy, self).__init__()
        
        # TODO: Initialize the network layers:
        #       - fc1: Linear(s_size, h_size) - First hidden layer
        #       - fc2: Linear(h_size, a_size) - Output layer
        #       - add any as you need.
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, h_size)
        self.fc3 = nn.Linear(h_size, a_size)

    def forward(self, x):
        """
        Forward pass through the network.
        
        Args:
            x: State tensor
            
        Returns:
            Action probabilities (after softmax)
        """
        # TODO: Implement forward pass:
        #       - Apply ReLU activation to first layer
        #       - Apply softmax to output layer for probability distribution
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        """
        Select action using the current policy.
        
        Args:
            state: Current state (numpy array)
            
        Returns:
            action: Selected action (int)
            log_prob: Log probability of selected action
        """
        # TODO: Convert state to tensor and add batch dimension
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        
        # TODO: Get action probabilities from forward pass
        probs = self.forward(state).cpu()
        
        # TODO: Create categorical distribution and sample action
        m = Categorical(probs)
        action = m.sample()
        
        # TODO: Return action and its log probability
        return action.item(), m.log_prob(action)

# TODO: Test the policy network architecture
n_inputs = env.observation_space.shape[0]
n_outputs = env.action_space.n
h_size = 64

test_policy = Policy(n_inputs, n_outputs, h_size).to(device)
print(f"Policy network created with {sum(p.numel() for p in test_policy.parameters())} parameters")

## 3️⃣ Hyperparameters and Training Setup

**Task**: Set up hyperparameters and initialize the training components.

**Requirements**:
- Define appropriate hyperparameters for stable REINFORCE training
- Initialize policy network with proper architecture
- Set up optimizer with suitable learning rate
- Configure training parameters (episodes, max steps, etc.)

In [ ]:
# TODO: Define hyperparameters:
hyperparameters = {
    "h_size": 64,                    # Hidden layer size
    "n_training_episodes": 2000,      # Total training episodes
    "n_evaluation_episodes": 10,      # Episodes for evaluation
    "max_t": 1000,                    # Maximum steps per episode
    "gamma": 0.99,                    # Discount factor for returns
    "lr": 1e-3,                       # Learning rate for policy optimization
    "env_id": "LunarLander-v3",       # Environment identifier
    "state_space": env.observation_space.shape[0],  # State dimension (8)
    "action_space": env.action_space.n,             # Action dimension (4)
}


# TODO: Initialize policy network and optimizer:
#       - Create policy network with specified architecture
#       - Use Adam optimizer with specified learning rate
policy = Policy(hyperparameters["state_space"], 
                hyperparameters["action_space"], 
                hyperparameters["h_size"]).to(device)

optimizer = optim.Adam(policy.parameters(), lr=hyperparameters["lr"])

## 4️⃣ REINFORCE Algorithm Implementation

**Task**: Implement the core REINFORCE training function.

**Requirements**:
- Collect complete episodes using current policy
- Calculate Monte Carlo returns with discount factor
- Implement return normalization for training stability
- Compute policy gradient loss using log probabilities
- Perform policy parameter updates using gradient ascent
- Track training progress and episode statistics

In [ ]:
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    """
    REINFORCE algorithm implementation.
    
    Args:
        policy: Policy network to train
        optimizer: Optimizer for policy parameters
        n_training_episodes: Number of training episodes
        max_t: Maximum steps per episode
        gamma: Discount factor for returns
        print_every: Frequency of progress printing
        
    Returns:
        scores: List of episode scores
    """
    # TODO: Initialize tracking variables:
    #       - scores_deque: Rolling window of recent scores
    #       - scores: Complete list of all episode scores
    scores_deque = deque(maxlen=100)
    scores = []
    
    # TODO: Main training loop over episodes
    for i_episode in range(1, n_training_episodes + 1):
        # TODO: Initialize episode-specific variables:
        #       - saved_log_probs: Store log probabilities of actions taken
        #       - rewards: Store rewards received during episode
        saved_log_probs = []
        rewards = []
        
        # TODO: Reset environment and handle potential tuple return
        state = env.reset()
        if isinstance(state, tuple):
            state = state[0]  # Handle tuple return from env.reset()
        
        # TODO: Episode execution loop
        for t in range(max_t):
            # TODO: Select action using current policy
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            
            # TODO: Execute action in environment
            step_result = env.step(action)
            
            # TODO: Handle different return formats from environment
            if len(step_result) == 5:
                state, reward, done, truncated, _ = step_result
                done = done or truncated
            else:
                state, reward, done, _ = step_result
            
            # TODO: Store reward and check for episode termination
            rewards.append(reward)
            if done:
                break
        
        # TODO: Calculate episode statistics
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # TODO: Calculate Monte Carlo returns (discounted cumulative rewards):
        #       - Work backwards from end of episode
        #       - Apply discount factor gamma to future returns
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        
        # Calculate returns by working backwards through the episode
        for t in range(n_steps)[::-1]:
            disc_return_t = (returns[0] if len(returns) > 0 else 0)
            returns.appendleft(gamma * disc_return_t + rewards[t])

        # TODO: Normalize returns for training stability:
        #       - Convert to tensor and normalize (zero mean, unit variance)
        #       - Add small epsilon to prevent division by zero
        eps = np.finfo(np.float32).eps.item()
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # TODO: Calculate policy loss:
        #       - REINFORCE uses negative log probability weighted by return
        #       - Sum over all timesteps in the episode
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        # TODO: Perform optimization step:
        #       - Clear gradients, compute gradients, update parameters
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        # TODO: Print progress at specified intervals
        if i_episode % print_every == 0:
            print('Episode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_deque)))

    return scores

## 5️⃣ Training Execution

**Task**: Execute the REINFORCE training process.

**Requirements**:
- Run training for the specified number of episodes
- Monitor training progress and convergence
- Track episode scores and performance metrics
- Handle training time efficiently (this may take 15-30 minutes)

In [ ]:
# TODO: Execute training with progress tracking
print("Starting REINFORCE training...")
print("="*50)

# TODO: Train the agent using REINFORCE algorithm
scores = reinforce(policy, 
                   optimizer, 
                   hyperparameters["n_training_episodes"], 
                   hyperparameters["max_t"], 
                   hyperparameters["gamma"], 
                   100)  # Print every 100 episodes

print("Training completed!")
print(f"Final average score over last 100 episodes: {np.mean(scores[-100:]):.2f}")

## 6️⃣ Training Results Visualization

**Task**: Create comprehensive visualizations of the training progress.

**Requirements**:
- Plot episode scores over training
- Analyze learning curves and convergence

In [ ]:
# TODO: Create comprehensive training visualization
plt.figure(figsize=(12, 6))

# TODO: Plot raw scores
plt.plot(scores, alpha=0.6, color='blue', label='Episode Scores')


# TODO: Add horizontal line at target performance (200 points)
plt.axhline(y=200, color='green', linestyle='--', linewidth=2, label='Target Score (200)')

# TODO: Customize plot appearance
plt.title('REINFORCE Training Progress on LunarLander-v3', fontsize=14, fontweight='bold')
plt.xlabel('Episode')
plt.ylabel('Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7️⃣ Agent Testing and Video Generation

**Task**: Test the trained agent and create a video visualization.

**Requirements**:
- Test the trained agent using the learned policy (no exploration)
- Record the agent's performance in the environment
- Create a video of the agent playing
- Analyze the agent's final performance

In [ ]:
# TODO: Set up video recording environment
from gymnasium.wrappers import RecordVideo
from IPython.display import HTML
from IPython import display
import glob
import base64, io, os
import gymnasium as gym

# Set environment variable for headless rendering
os.environ['SDL_VIDEODRIVER'] = 'dummy'

def show_video():
    """Display recorded video in Jupyter notebook."""
    mp4list = glob.glob('video*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display.display(HTML(data='''<video alt=\"Trained REINFORCE Agent\" autoplay 
                loop controls style=\"height: 400px;\">
                <source src=\"data:video/mp4;base64,{0}\" type=\"video/mp4\" />
             </video>'''.format(encoded.decode('ascii'))))
    else: 
        print("Could not find video")

def wrap_env(env):
    """Wrap environment for video recording."""
    env = RecordVideo(env, video_folder="./", episode_trigger=lambda x: True, name_prefix="video")
    return env

# TODO: Test the trained agent
print("Testing Trained REINFORCE Agent...")
print("="*40)

# TODO: Create test environment with rendering
env_test = gym.make("LunarLander-v3", render_mode="rgb_array")
vid = wrap_env(env_test)

# TODO: Run test episode
state, info = vid.reset()
total_reward = 0
episode_steps = 0

for t in range(1000):  # Maximum steps per episode
    # TODO: Use trained policy (REINFORCE agent) - deterministic action selection
    with torch.no_grad():
        state_tensor = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = policy.forward(state_tensor).cpu()
        action = torch.argmax(probs).item()  # Greedy action selection for testing
    
    # TODO: Execute action
    observation, reward, terminated, truncated, _ = vid.step(action)
    total_reward += reward
    episode_steps += 1
    
    # TODO: Check if episode finished
    if terminated or truncated:
        print(f"Episode finished after {episode_steps} timesteps")
        print(f"Total reward: {total_reward:.2f}")
        
        break   
    
    # TODO: Update state
    state = observation

# TODO: Clean up
vid.close()

print(f"\nTrained agent performance: {total_reward:.2f} points")

## 8️⃣ Display Training Video

**Task**: Show the recorded video of the trained agent.

**Requirements**:
- Display the video inline in the notebook
- Verify that the agent has learned effective landing behavior

In [ ]:
# TODO: Display the recorded video
show_video()

## 9️⃣ Action Selection Analysis

**Task**: Analyze the trained agent's action selection patterns and decision-making.

**Requirements**:
- Track action frequencies during test episodes
- Visualize action distribution patterns
- Understand the agent's learned strategy
- Compare action selection across different states

In [ ]:
# TODO: Analyze action selection patterns
print("Analyzing Action Selection Patterns...")
print("="*45)

action_names = ['Do Nothing', 'Fire Left', 'Fire Main', 'Fire Right']
action_counts = {name: 0 for name in action_names}
state_action_history = []

# TODO: Run episode and track all actions
env_test = gym.make("LunarLander-v3")
state, info = env_test.reset()

for t in range(1000):
    # TODO: Get action probabilities and select action using REINFORCE policy
    state_tensor = torch.from_numpy(state).float().unsqueeze(0).to(device)
    with torch.no_grad():
        probs = policy.forward(state_tensor).cpu()
        action, _ = policy.act(state)
    
    # TODO: Track action and state
    action_counts[action_names[action]] += 1
    state_action_history.append({
        'step': t,
        'state': state,
        'action': action,
        'action_probs': probs.numpy().flatten()
    })
    
    # TODO: Execute action
    step_result = env_test.step(action)
    if len(step_result) == 5:
        observation, reward, terminated, truncated, _ = step_result
        done = terminated or truncated
    else:
        observation, reward, terminated, _ = step_result
        done = terminated
    
    if done:
        break
        
    state = observation

env_test.close()

# TODO: Visualize action distribution
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.title('Action Selection Frequency', fontsize=14, fontweight='bold')
actions = list(action_counts.keys())
counts = list(action_counts.values())
colors = ['red', 'blue', 'orange', 'green']
bars = plt.bar(actions, counts, color=colors, alpha=0.7)
plt.ylabel('Frequency')
plt.xticks(rotation=45)

# TODO: Add value labels on bars
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(count), ha='center', va='bottom', fontweight='bold')

plt.grid(True, alpha=0.3)

# TODO: Plot action selection over time
plt.subplot(2, 2, 2)
plt.title('Action Selection Over Time', fontsize=14, fontweight='bold')
steps = [entry['step'] for entry in state_action_history]
actions_taken = [entry['action'] for entry in state_action_history]
plt.scatter(steps, actions_taken, alpha=0.6, c=actions_taken, cmap='viridis')
plt.ylabel('Action')
plt.xlabel('Time Step')
plt.yticks([0, 1, 2, 3], action_names)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📋 Assignment Evaluation Criteria

Your REINFORCE homework will be evaluated based on the following criteria:

### **Implementation Correctness (40%)**
- ✅ Proper REINFORCE algorithm implementation
- ✅ Correct Monte Carlo return calculation with discounting
- ✅ Appropriate policy network architecture and forward pass
- ✅ Proper policy gradient computation and optimization
- ✅ Return normalization for training stability

### **Training Performance (25%)**
- ✅ Agent trains without errors for specified episodes
- ✅ Achieves reasonable performance (average reward > 100) after training
- ✅ Shows clear learning progress over time
- ✅ Proper use of hyperparameters

### **Code Quality and Documentation (20%)**
- ✅ Clean, readable code with comprehensive comments
- ✅ Proper tensor handling and device management
- ✅ Efficient implementation without memory leaks
- ✅ Well-structured functions and classes

### **Analysis and Understanding (15%)**
- ✅ Comprehensive training visualizations
- ✅ Thoughtful action selection analysis